# `vectorCube` - Blueprint for processing vector data cubes

In this tutorial we will discuss the Abstract Base Class `vectorCube`, which serves as the overarching blueprint in the vector data processing pipeline. The class can be visualized as the following UML diagram

```mermaid
classDiagram
    class dataCube {
        +logger
        +initialize_pipeline(recipe, dataset_name, logger) Tuple
        #_setup_pipeline_logger(logger_name, log_filepath, level) Logger
    }

    class vectorEngine {
        <<Engine Component>>
        +build_safe_fetch_envelope()
        +coordinate_to_geometry()
        +map_cellCollection_to_template()
        +map_points_to_template()
        +map_point_cloud_to_template()
        +map_polygon_to_template()
        +validate_vector_transformation()
        +sanitize_geometries()
    }

    class vectorCube {
        <<abstract>>
        +fetch_vector_by_bbox(file_path, target_grid_name, target_bbox, ...)
        +fetch_data(recipe, logger)* GeoDataFrame
        +resolve_target_grid(spatial_cfg, logger)* str
        +process_cube(recipe, dataset_name, logger, ...) Item
        +aggregate_vector_cube(data, recipe, dataset_name, logger) DataFrame
        +generate_vector_stac_item(df, recipe, dataset_name, ...) Item
        #_apply_cf_temporal_standards(df, time_cols)
    }

    class vectorDatasourceCube {
        <<Concrete Adapter>>
        +fetch_data(recipe, logger, downloaded_filepath, ...) GeoDataFrame
        +resolve_target_grid(spatial_cfg, logger) str
        +generate_query_from_recipe(recipe, logger) str
        #_mine_crs_from_grid(df, logger) str
        #_parse_cellcode_to_polygon(code) Polygon
        #_validate_local_data(df, source_cfg, logger)
    }

    %% Relationships
    dataCube <|-- vectorCube : Inherits Orchestration
    vectorEngine <|-- vectorCube : Inherits Geometry Math
    vectorCube <|-- vectorDatasourceCube : Implements Abstract Methods
```

The Vector Cube ecosystem is designed around a modular, object-oriented hierarchy that strictly separates spatial mathematics, pipeline orchestration, and data ingestion. 

1) `dataCube` (The Operational Base)
The `dataCube` acts as the foundational telemetry and file-system manager. It is completely agnostic to whether the data is raster or vector.
    * **Responsibilities:** Provisions output directories, initializes tracking loggers, and boots hardware resource profilers.
    * **Key Methods:** `initialize_pipeline()`, `_setup_pipeline_logger()`.

2) `vectorEngine` (The Spatial Powerhouse)
The `vectorEngine` is a lower-level mixin class containing all the complex geometric math and geospatial routing logic. 
    * **Responsibilities:** Handles spatial intersections, sanitizes broken topologies, and translates arbitrary coordinates into points, polygons, or probabilistic point clouds.
    * **Key Methods:** `coordinate_to_geometry()`, `map_geometry_to_template()`.

3) `vectorCube` (The Abstract Orchestrator)
The `vectorCube` acts as the master bridge. It inherits hardware tracking from `dataCube` and spatial math from `vectorEngine` to orchestrate the end-to-end vector pipeline. 
    * **Responsibilities:** Drives the core `process_cube()` loop, executes fractional grouping/aggregations via `aggregate_vector_cube()`, and packages the final Parquet outputs into PySTAC Items. 
    * **Abstract Contracts:** It deliberately leaves `fetch_data()` and `resolve_target_grid()` blank (`@abstractmethod`), forcing child classes to define how their specific data is acquired.

4) `vectorDatasourceCube` (The Concrete Adapter)
This is the specialized child class (like our generalized GBIF adapter) that brings the pipeline to life. 
    * **Responsibilities:** Translates pipeline YAML recipes into vendor-specific API calls or SQL queries, normalizes source-specific schemas, and fulfills the abstract contracts mandated by `vectorCube`.
    * **Key Methods:** `fetch_data()`, `resolve_target_grid()`, `generate_query_from_recipe()`.

## Abstract methods of the `vectorCube` class

The upcoming tutorial will guide you through creating your own `vectorDatasourceCube` by extending the `vectorCube` base class. The primary focus will be successfully implementing the mandatory abstract methods to hook a new API or database into the core spatial engine.

* **Implementing `fetch_data()`:**
  * How to translate a YAML recipe block into an exact data retrieval payload.
  * Handling synchronous vs. asynchronous downloads and caching.
  * Converting raw JSON or CSV coordinate tables into standardized `geopandas.GeoDataFrame` objects ready for the engine.

* **Implementing `resolve_target_grid()`:**
  * Mapping arbitrary user inputs (e.g., `"10km"`) to the engine's strict `GRID_REGISTRY` constraints.
  * Writing defensive fallback logic to prevent projection misalignment.

* **Validating Schemas & Pre-processing:**
  * Building internal validators (like `_validate_local_data()`) to ensure required columns (e.g., coordinates, uncertainty metrics) exist before the spatial engine spins up.
  * Dynamically parsing vendor-specific artifacts (such as translating custom grid cell codes into Shapely polygons).

* **Testing the Full `process_cube()` Lifecycle:**
  * Feeding your custom connector into the master loop to watch it shatter spatial topologies, group relational metrics, and output a valid STAC Item.

### `fetch_data`

The `fetch_data` method acts as the primary Input/Output (I/O) gateway connecting the engine's core spatial mathematics to the real-world data sources.

#### Design and Role
In the abstract `vectorCube` base class, `fetch_data()` is deliberately left undefined (`@abstractmethod`). This design forces concrete child classes (like the GBIF adapter) to implement their own distinct retrieval logics while honoring a strict contract: **they must return a standardized `geopandas.GeoDataFrame`**.

#### Functionality within the Concrete Adapter (e.g., GBIF)
When implemented in a concrete child class, `fetch_data` is responsible for:
*   **Asynchronous Coordination:** In the specific case of GBIF we have to deal with an asynchronous API where we submit a SQL query and have to wait for the download it already. In the standard case one would immediately encode a data retrieval step in this function. However due to it being an abstract method we can define it to the specific requirements of the datasource. The fetch method in this case will ping the download status and initiate it when ready or read in the data if an input filepath is provided in the yaml recipe
*   **Dynamic Payload Parsing:** It natively handles various data structures, safely extracting tabular coordinates from compressed Zipped archives, CSVs, or Parquet files.
*   **Schema Normalization and Validation:** It scans the user-provided data and cross-references it against the pipeline's YAML requirements. This includes verifying mandatory coordinate columns (e.g., `latitude`/`longitude`), ensuring required grouping dimensions exist, and even mapping column casings dynamically so the engine doesn't crash during aggregation.
*   **Geometric Construction:** Ultimately, it maps the raw tabular coordinates into active geometry objects (`POINTS`, `POLYGONS`, etc.) required by the spatial intersection engines downstream.

### `resolve_target_grid`

The `resolve_target_grid` method acts as the critical spatial bridge between abstract user requests in the YAML recipe and the exact mathematical grid templates hardcoded into the engine.

#### Design and Role
Just like data ingestion, `resolve_target_grid()` is defined as an `@abstractmethod` within the `vectorCube` base class. This design choice forces every concrete child class to interpret the user's spatial configuration according to the conventions of that specific datasource. The strict contract here is that the method **must return a validated string** representing a master grid key recognized by the engine.

#### Functionality within the Concrete Adapter (e.g., GBIF)
When implemented in a concrete child class like `gbifCube`, `resolve_target_grid` takes on several key responsibilities:
*   **Dynamic Key Assembly:** It parses the `spatial` configuration block to extract the base grid (e.g., "EEA") and resolution (e.g., "1km"). It then intelligently checks if the user provided an exact match (like "EEA_1km") or pieces them together to construct the proper key.
*   **Safe Defaults and Fallbacks:** If the recipe entirely omits a target grid, the adapter is responsible for stepping in and assigning a safe default. For example, the GBIF adapter automatically defaults to "Global_WGS84_30sec" if no target is specified.
*   **Strict Registry Validation:** Before returning the string, it strictly verifies that the constructed key actually exists inside the engine's internal `GRID_REGISTRY`. This guarantees that downstream processes will have access to perfectly aligned pixel resolutions and bounding boxes.
*   **Fail-Safe Error Handling:** If a user requests a grid or resolution that the engine does not support, the method acts as a gatekeeper, raising a descriptive `KeyError` that lists available valid grids rather than allowing the pipeline to crash blindly during spatial math operations.

## Cube processing pipeline

```mermaid
flowchart TD
    %% Define Nodes
    Start([Start process_cube])
    Init[1. Initialize Pipeline & Loggers]
    Fetch[2. fetch_data & Sanitize Geometries]
    CheckRaw{Mode == 'raw'?}
    RawExport[Export Raw Parquet]
    GridResolve[3. Resolve Target Grid & BBox]
    SpatialRoute[4. Dynamic Spatial Routing]
    QAQC[5. QA/QC Validation & Export Unaggregated]
    CheckAgg{Has Aggregate Config?}
    Aggregate[6. aggregate_vector_cube]
    STAC[7. generate_vector_stac_item]
    End([End])

    %% Define Subgraphs for Clarity
    subgraph Data Acquisition
        Init --> Fetch
        Fetch --> CheckRaw
    end

    subgraph Spatial Engine
        CheckRaw -- No --> GridResolve
        GridResolve --> SpatialRoute
        SpatialRoute --> QAQC
    end

    subgraph Metrics & Metadata
        QAQC --> CheckAgg
        CheckAgg -- Yes --> Aggregate
        Aggregate --> STAC
        CheckAgg -- No --> STAC
    end

    %% Connections
    Start --> Init
    CheckRaw -- Yes --> RawExport --> End
    STAC --> End

The `process_cube` method is the central nervous system of the `vectorCube` framework. It directs the flow of data from raw API ingestion all the way to standardized, analysis-ready STAC catalogs. 

Here is the step-by-step breakdown of the pipeline and how aggregation and STAC generation fit into the architecture:

### 1. Pipeline Initialization
The orchestrator kicks off by calling `initialize_pipeline()`. This provisions the specific output directories, parses the baseline recipe configurations, and boots the dual-stream loggers and hardware resource profilers. 

### 2. Fetch & Build Base Topology Data
The pipeline invokes the abstract `fetch_data()` method to trigger the vendor-specific data retrieval (e.g., executing a GBIF SQL query). 
*   **Raw Mode Ejection:** If the user requested `processing_mode: "raw"`, the pipeline immediately saves the unmapped data to disk and halts further geometric processing. 
*   **Sanitization:** For standard modes, it cleans the geometries by dropping empty records and forcing valid 2D planar topologies.

### 3. Resolve Master Grid & Bounding Box
The engine translates the user's spatial recipe into a physical template by calling `resolve_target_grid()`. It then dynamically calculates a safe bounding box buffered by the exact target resolution to prevent boundary edge starvation when pulling geometries.

### 4. Dynamic Spatial Routing (The Mapping Phase)
Based on the `topology` defined in the YAML recipe (Point, Polygon, or Point Cloud), the orchestrator routes the dataset to the appropriate spatial intersection engine. This maps the continuous spatial coordinates to discrete template grid cells (either 1-to-1 classifications or 1-to-N fractional splits).

### 5. Dynamic QA/QC & Unaggregated Export
Before summarizing, the pipeline validates the transformation to ensure spatial mass was conserved during mapping (preventing data leaks). It then exports this foundational relational table as the `_unaggregated.parquet` file, preserving the raw linkage between source geometries and assigned grid cells.

### 6. Aggregation (`aggregate_vector_cube`)
If an `aggregate` block is provided in the recipe, the `aggregate_vector_cube()` method takes over. 
*   **Functionality:** It groups the unaggregated data by the requested dimensions (e.g., `year`, `month`, and the spatial `grid_idx`) and computes user-defined metrics like sums, means, or unique counts. 
*   **Fractional Weighting:** Crucially, if the data underwent fractional mapping, this method automatically multiplies additive metrics (like `sum` or `mean`) by the `areal_fraction` to strictly conserve probability mass across split cells. 
*   **Output:** The result is exported as the highly compressed `_aggregated.parquet` file.

### 7. STAC Item Generation (`generate_vector_stac_item`)
To make the resulting vector cube discoverable and interoperable, the pipeline terminates by building a PySTAC Item. 
*   **Functionality:** It extracts temporal bounds and processing provenance directly from the data and the recipe. It generates a dedicated "Spatial Dimension Table" mapping abstract grid indices to physical WKT polygons. 
*   **Asset Linking:** Finally, it bundles all the generated parquet files (source geometries, unaggregated fractions, aggregated cube, and spatial dimensions) as linked assets within a single `stac.json` file.

## The `vector_processing` Configuration Block

The `vector_processing` block in your YAML recipe strictly dictates how the mathematical spatial engine will interpret, shape, and assign your raw geometries to the master target grid.

```
vector_processing:
      # Desired geometric topology: "point", "polygon", or "point_cloud".
      topology: ""
      
      # Mapping output: "classification" (1-to-1 cell assignment) or 
      # "fractional" (1-to-N probability/areal weights).
      mapping_mode: ""  
      
      # Spatial lookup strategy: "intersect" (precise overlap) or "kdtree" (nearest centroid).
      # Note: KDTree is only mathematically valid when mapping_mode is 'classification'.
      spatial_method: ""

      # Topology-specific configurations passed directly to the geometry engines.
      # The orchestrator will only apply the config matching the active 'topology'.
      topology_config:
        point_cloud:
          # Number of randomized points generated in the point cloud
          n_passes: 100
          
          # Underlying probability distribution (either "gaussian" or "uniform") 
          # with a width determined by the uncertainty radius
          distribution: "gaussian"
          
          # Random seed generator to ensure reproducibility
          random_seed: 42

        polygon:
          # Number of quadrant segments of the polygon determining the smoothness of the polygon being constructed.
          # 2-4 is very coarse, 16-32 is very smooth, 8 is the default value in many libraries.
          # Note: Increasing this will increase the impact on RAM memory during intersection operations.
          quad_segs: 8
```


### A. `topology` (Geometric Representation)
This field defines the physical shape constructed from your raw point coordinates.
*   **`point`**: A simple 2D coordinate representation with zero area.
*   **`polygon`**: Projects the coordinate uncertainty radius outward into a solid geometric buffer.
*   **`point_cloud`**: A probabilistic array of randomized points scattered around the origin.

### B. `mapping_mode` (The Assignment Rule)
This defines how the spatial mass of your topology is distributed across the grid.
*   **`classification`**: Executes a 1-to-1 mapping. The observation is assigned entirely to a single "winning" grid cell.
*   **`fractional`**: Executes a 1-to-N mapping. The observation is mathematically shattered, returning probabilistic weights (or area percentages) for every grid cell it touches.

### C. `spatial_method` (The Mathematical Engine)
*   **`intersect`**: Executes rigorous spatial overlap calculations. It analyzes exact polygon intersections or counts exact point cloud hits inside physical cell boundaries.
*   **`kdtree`**: An optimized nearest-neighbor lookup that snaps the geometry to the closest cell centroid. *(Note: KDTree cannot be used for `fractional` mapping, as it inherently forces a single 1-to-1 classification).*

### D. `topology_config` (Shape Tuning Parameters)
This nested dictionary passes hyper-parameters directly to the geometric generators. The engine dynamically routes to the configuration matching your active `topology` selection.

*   **`point_cloud` configuration:**
    *   **`n_passes`**: The total number of artificial points to scatter into the cloud.
    *   **`distribution`**: Defines the spread pattern (`gaussian` or `uniform`) constrained by the observation's spatial uncertainty radius.
    *   **`random_seed`**: Ensures the randomized cloud generation remains strictly reproducible.

*   **`polygon` configuration:**
    *   **`quad_segs`**: Controls the smoothness of the generated circular buffer. A lower number creates coarse, blocky approximations, while a higher number creates perfectly smooth circles at the cost of heavier RAM usage during the geometric intersection routines.

## The aggregate configuration block

This guide details the `aggregate` configuration block within your YAML recipe. This block dictates how the engine condenses the raw, shattered spatial mappings into a final, multidimensional statistical cube.

## The `aggregate` Configuration Block

Once the spatial engine completes its geometric intersections, the pipeline references this block to execute relational groupings and mathematical summaries.

```
aggregate:
      export_unaggregated: true
      
      # Defines the non-spatial dimensions of the final data cube.
      # The spatial cell ID (grid_idx) is automatically appended by the engine.
      group_by_columns: ["", ""]
      
      metrics:
        - column: ""
          method: ""
          weighted: false
          rename: ""
```


### A. `export_unaggregated` (Boolean)
Before summarizing the data, you can choose to save the raw relational mapping.
*   **`true` / `false`**: If set to `true`, the pipeline exports a highly granular `_unaggregated.parquet` file. This table retains every individual source record explicitly linked to its assigned grid cell(s) alongside its fractional weights, which is invaluable for debugging or custom downstream processing.

### B. `group_by_columns` (Dimensionality)
This defines the non-spatial dimensions of your final data cube.
*   **List of strings**: You define which attributes to group the data by (e.g., `["year", "month"]`).
*   *Note:* The engine automatically appends the spatial cell identifier (`grid_idx`) to this list under the hood, ensuring the output is always a spatial cube.

### C. `metrics` (Statistical Operations)
This is a list of dictionaries defining the exact calculations to perform on your grouped data. Each metric requires the following parameters:

*   **`column`**: The specific attribute column from your raw data to run the math on (e.g., `"areal_fraction"`, `"speciesKey"`, `"recordedBy"`).
*   **`method`**: The Pandas/SQL statistical operation to execute. Common methods include:
    *   `sum`: Adds the values together.
    *   `nunique`: Counts the distinct, unique occurrences (perfect for species richness or distinct observers).
    *   `mean`, `min`, `max`, `count`.
*   **`weighted` (Boolean)**: Crucial for fractional mapping modes. 
    *   If `true`, the engine automatically multiplies the column's values by the calculated spatial fraction (`areal_fraction`) *before* executing additive methods like `sum` or `mean`. This ensures that probability mass and observation counts are strictly conserved across shattered cells.
    *   If `false`, the metric ignores the fractional weights (ideal for calculating things like distinct species richness where partial taxonomy doesn't make sense).
*   **`rename`**: The new column name for the resulting metric in the final aggregated dataset (e.g., `"expected_occurrences"`, `"species_richness"`).